In [4]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression, RANSACRegressor
from math import atan, degrees

def load_data(file_path):
    data = pd.read_csv(file_path, delim_whitespace=True, header=None, names=["t", "x", "y", "z"])
    return data[["t", "x", "y", "z"]]

df = load_data("ROVER1.GK")

plain = df[['x', 'y']]
height = df['z']

#--------------------------------------------------------------------МНК-----------------------------------------------------------------------
lr = LinearRegression()
lr.fit(plain, height)
a_mnk, b_mnk = lr.coef_
c_mnk = lr.intercept_

theta_mnk = atan(np.sqrt(a_mnk**2 + b_mnk**2)) # посчитали угол наклона через арктангенс
angle_mnk_degrees = degrees(theta_mnk)  # перевели в градусы

#--------------------------------------------------------------------РНК-----------------------------------------------------------------------
ransac = RANSACRegressor()
ransac.fit(plain, height)
a_rnk, b_rnk = ransac.estimator_.coef_
c_rnk = ransac.estimator_.intercept_

theta_rnk = atan(np.sqrt(a_rnk**2 + b_rnk**2))
angle_rnk_degrees = degrees(theta_rnk)

#----------------------------------------------------------------------------------------------------------------------------------------------
print(f"Угол наклона плоскости через МНК: {angle_mnk_degrees:.2f} градусов")
print(f"Угол наклона плоскости через РНК: {angle_rnk_degrees:.2f} градусов")

#--------------------------------------------------------------------РИСУЕМ--------------------------------------------------------------------
# Сетка
x_range = np.linspace(df['x'].min(), df['x'].max(), 10)
y_range = np.linspace(df['y'].min(), df['y'].max(), 10)
X_grid, Y_grid = np.meshgrid(x_range, y_range)

# Плоскости
Z_mnk = a_mnk * X_grid + b_mnk * Y_grid + c_mnk
Z_rnk = a_rnk * X_grid + b_rnk * Y_grid + c_rnk

fig = go.Figure()

fig.add_trace(go.Scatter3d(x=df['x'], y=df['y'], z=df['z'], mode='markers', marker=dict(size=2, color='blue'), name='Точки данных'))
fig.add_trace(go.Surface(z=Z_mnk, x=X_grid, y=Y_grid, colorscale='Reds', opacity=0.5, name='Плоскость МНК'))
fig.add_trace(go.Surface(z=Z_rnk, x=X_grid, y=Y_grid, colorscale='Greens', opacity=0.5, name='Плоскость РНК'))
fig.update_layout(title='Аппроксимация плоскости методом МНК и РНК', scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'))

fig.show()

Угол наклона плоскости через МНК: 0.71 градусов
Угол наклона плоскости через РНК: 0.25 градусов


<ipython-input-4-ba3da63d815b>:8: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead

